In [27]:
from huggingface_hub import notebook_login
notebook_login()

In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto" #For correct GPU placement
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [29]:
from datasets import load_dataset
data = load_dataset("fancyzhx/ag_news")
print (data)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [30]:

data["train"].to_pandas()["label"].value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

In [31]:
data['train'].features['label'].names

['World', 'Sports', 'Business', 'Sci/Tech']

In [32]:
import pandas as pd
def stratified_sampling(data,n, seed=42):
    df = data.to_pandas()
    n_per_class = n//4
    sampled = df.groupby('label',group_keys=False).apply(lambda x: x.sample(n=n_per_class,random_state=seed))
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)
train_5000_df = stratified_sampling(data['train'],5000)
train_500_df = stratified_sampling(data['train'],500)


C:\Users\Muhammad Sami\AppData\Local\Temp\ipykernel_4432\2964281954.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = df.groupby('label',group_keys=False).apply(lambda x: x.sample(n=n_per_class,random_state=seed))
C:\Users\Muhammad Sami\AppData\Local\Temp\ipykernel_4432\2964281954.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = df.groupby('label',group_keys=False).apply(lambda x: x.sample(n=

In [33]:
from datasets import Dataset

train_5000 = Dataset.from_pandas(train_5000_df, preserve_index=False)
train_500 = Dataset.from_pandas(train_500_df, preserve_index=False)


In [34]:
from datasets import ClassLabel

label_names = data['train'].features['label'].names
train_500 = train_500.cast_column("label", ClassLabel(names=label_names))
split = train_500.train_test_split(test_size=0.1, stratify_by_column="label", seed=42)

train_500_final = split["train"]
val_500 = split["test"]

train_5000 = train_5000.cast_column("label", ClassLabel(names=label_names))
split = train_5000.train_test_split(test_size=0.1, stratify_by_column="label", seed=42)

train_5000_final = split["train"]
val_5000 = split["test"]



Casting the dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [35]:
def tokenize_function(example):
    return tokenizer(example["text"] ,truncation=True,max_length=256)

tokenized_train_5000 = train_5000_final.map(tokenize_function, batched= True)
tokenized_train_500 = train_500_final.map(tokenize_function,batched = True)
tokenized_val_5000 = val_5000.map(tokenize_function,batched=True)
tokenized_val_500 = val_500.map(tokenize_function,batched=True)

    

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [36]:
for name, ds in [("train_500_final", tokenized_train_500), ("val_500", tokenized_val_500),
                  ("train_5000_final", tokenized_train_5000), ("val_5000", tokenized_val_5000)]:
    print(name, ds.column_names)

train_500_final ['text', 'label', 'input_ids', 'attention_mask']
val_500 ['text', 'label', 'input_ids', 'attention_mask']
train_5000_final ['text', 'label', 'input_ids', 'attention_mask']
val_5000 ['text', 'label', 'input_ids', 'attention_mask']


In [37]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [38]:
# 1. Stratified sample of ~1500 from the official test split
test_1500_df = stratified_sampling(data["test"], 1500)

# 2. Convert to Dataset + cast label to ClassLabel (needed if you plan to stratify further, 
#    and good practice for consistency even if not strictly required here)
from datasets import Dataset, ClassLabel

tokenized_test = Dataset.from_pandas(test_1500_df, preserve_index=False)
tokenized_test = tokenized_test.cast_column("label", ClassLabel(names=label_names))

# 3. Tokenize it the same way as everything else
tokenized_test = tokenized_test.map(tokenize_function, batched=True)

# 4. Sanity check
print(tokenized_test.column_names)
print(len(tokenized_test))
print(tokenized_test["label"][:10])  # should look reasonably mixed across 0-3

C:\Users\Muhammad Sami\AppData\Local\Temp\ipykernel_4432\2964281954.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = df.groupby('label',group_keys=False).apply(lambda x: x.sample(n=n_per_class,random_state=seed))


Casting the dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

['text', 'label', 'input_ids', 'attention_mask']
1500
[2, 3, 1, 1, 1, 2, 2, 1, 3, 0]


In [39]:
from transformers import AutoModelForSequenceClassification
label_names = data['train'].features['label'].names
id2label = {i:name for i,name in enumerate(label_names)}
label2id = {name:i for i,name in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label = id2label,
    label2id = label2id,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Some parameters are on the meta device because they were offloaded to the cpu.


In [40]:
#Counting trainable parameters
def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.2f}%)")
    return trainable, total

count_trainable_params(model)

Trainable: 1,543,720,448 / Total: 1,543,720,448 (100.00%)


(1543720448, 1543720448)

In [41]:
#Vram tracking functions
import torch

def reset_vram():
    torch.cuda.reset_peak_memory_stats()

def get_peak_vram_mb():
    return torch.cuda.max_memory_allocated() / (1024 ** 2)

In [42]:
#Metrics for the  trainer
import numpy as np
import evaluate



accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

In [20]:
#Training arguments for the trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results/full-ft-500",   
    eval_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,          
    num_train_epochs=5,                     
    learning_rate=2e-5,                     
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=False,
    metric_for_best_model="f1",
    seed=42,                                
    report_to="none",                       
    bf16=True,                              # mixed precision, saves VRAM/time
    gradient_checkpointing=True, 
               
)

In [21]:
#Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_500,
    eval_dataset=tokenized_val_500,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,   
)

In [22]:
#Full fine-tuning of the model
import time

reset_vram()
start = time.time()

trainer.train()

elapsed = time.time() - start
peak_vram = get_peak_vram_mb()
trainable, total = count_trainable_params(model)

print(f"Time: {elapsed:.1f}s | Peak VRAM: {peak_vram:.0f}MB | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,3.340314,0.598602,0.800000,0.801599
2,0.408760,0.416225,0.800000,0.800705
3,0.078943,0.473205,0.900000,0.901864
4,0.016741,0.438118,0.920000,0.920705
5,0.007453,0.434602,0.920000,0.920705


Trainable: 1,543,720,448 / Total: 1,543,720,448 (100.00%)
Time: 1312.7s | Peak VRAM: 14748MB | Trainable: 1,543,720,448 (100.00%)


In [23]:
test_results_5000 = trainer.evaluate(tokenized_test)
print("TEST SET RESULTS (full-FT, 5000 samples):", test_results_5000)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.007453,0.587487,5,0.846667,0.846846


TEST SET RESULTS (full-FT, 5000 samples): {'eval_loss': 0.5874865055084229, 'eval_accuracy': 0.8466666666666667, 'eval_f1': 0.8468457197817454}


In [24]:
#Reloading a fresh model
import gc
del model
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    torch_dtype="auto",
    device_map="auto"
)


from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results/full-ft-5000",
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=20,
    metric_for_best_model="f1",
    seed=42,
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
)

# 3. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_5000,
    eval_dataset=tokenized_val_5000,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# 4. Train with measurement harness
reset_vram()
start = time.time()

trainer.train()

elapsed = time.time() - start
peak_vram = get_peak_vram_mb()
trainable, total = count_trainable_params(model)

print(f"Time: {elapsed:.1f}s | Peak VRAM: {peak_vram:.0f}MB | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Some parameters are on the meta device because they were offloaded to the cpu.
[transformers] The model is already on multiple devices. Skipping the move to device specified in `args`.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.171921,0.204825,0.938000,0.938135
2,0.478079,0.232747,0.934000,0.934012
3,0.146035,0.269271,0.934000,0.933958


Trainable: 1,543,720,448 / Total: 1,543,720,448 (100.00%)
Time: 4779.6s | Peak VRAM: 13080MB | Trainable: 1,543,720,448 (100.00%)


In [25]:
# 5. Evaluate on the fixed held-out test set — do this now too, so it's ready when you check in the morning
test_results = trainer.evaluate(tokenized_test)
print("TEST SET RESULTS:", test_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.146035,0.350156,3,0.922000,0.921968


TEST SET RESULTS: {'eval_loss': 0.3501560091972351, 'eval_accuracy': 0.922, 'eval_f1': 0.9219683133962104}


In [61]:
#Fresh base model for LoRA from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [62]:
#Lora config
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,          # sequence classification
    r=16,                                 # rank — controls adapter capacity
    lora_alpha=32,                        # scaling factor, commonly 2x the rank
    lora_dropout=0.1,                     # regularization on the adapters
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen2's attention projections
    bias="none",
)

In [63]:
#wrapping the model around the LoRA configuration
model = get_peft_model(model, lora_config)

In [21]:
#Getting trainable parameters
model.print_trainable_parameters()

trainable params: 4,364,288 || all params: 1,548,084,736 || trainable%: 0.2819


In [64]:
#Fine tunign the model using LoRA
from transformers import TrainingArguments, EarlyStoppingCallback, Trainer


torch.cuda.empty_cache()
training_args = TrainingArguments(
    output_dir="./results/lora-500",
    eval_strategy="epoch",
    save_strategy="no",
    gradient_checkpointing=True,
    load_best_model_at_end=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,          # higher LR than full-FT 
    weight_decay=0.01,
    logging_steps=10,
    metric_for_best_model="f1",
    seed=42,
    report_to="none",
    bf16=True,
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_500,
    eval_dataset=tokenized_val_500,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


reset_vram()
start = time.time()
trainer.train()
elapsed = time.time() - start

peak_vram = get_peak_vram_mb()
trainable, total = count_trainable_params(model)
print(f"Time: {elapsed:.1f}s | Peak VRAM: {peak_vram:.0f}MB | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

test_results_lora_500 = trainer.evaluate(tokenized_test)
print("TEST SET RESULTS (LoRA, 500 samples):", test_results_lora_500)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,4.480616,0.461150,0.860000,0.858271
2,1.296208,0.405718,0.860000,0.858056
3,0.476028,0.368987,0.820000,0.817500


Trainable: 4,364,288 / Total: 1,548,084,736 (0.28%)
Time: 265.0s | Peak VRAM: 5050MB | Trainable: 4,364,288 (0.28%)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.476028,0.433445,3,0.862000,0.863192


TEST SET RESULTS (LoRA, 500 samples): {'eval_loss': 0.43344464898109436, 'eval_accuracy': 0.862, 'eval_f1': 0.8631922644147334}


In [66]:
#LoRA on the training set of 5000 inputs
from transformers import AutoModelForSequenceClassification, TrainingArguments, EarlyStoppingCallback, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# Fresh base model
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    num_labels=4, id2label=id2label, label2id=label2id,
    torch_dtype="auto", device_map="auto"
)

# Reapply LoRA (same config as before)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()  # confirm 0.28% before proceeding

# New training args
training_args = TrainingArguments(
    output_dir="./results/lora-5000",
    eval_strategy="epoch", save_strategy="no", load_best_model_at_end=False,
    per_device_train_batch_size=4, per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, num_train_epochs=3,
    learning_rate=2e-4, weight_decay=0.01, logging_steps=20,
    metric_for_best_model="f1", seed=42, report_to="none",
    bf16=True, gradient_checkpointing=True,
)

# New trainer
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_train_5000, eval_dataset=tokenized_val_5000,
    data_collator=data_collator, processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

reset_vram()
start = time.time()
trainer.train()
elapsed = time.time() - start

peak_vram = get_peak_vram_mb()
trainable, total = count_trainable_params(model)
print(f"Time: {elapsed:.1f}s | Peak VRAM: {peak_vram:.0f}MB | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

test_results = trainer.evaluate(tokenized_test)
print("TEST SET RESULTS (LoRA, 5000 samples):", test_results)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 4,364,288 || all params: 1,548,084,736 || trainable%: 0.2819


[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.182681,0.194640,0.932000,0.931723
2,0.931478,0.226436,0.924000,0.923655
3,0.301016,0.282961,0.942000,0.942078


Trainable: 4,364,288 / Total: 1,548,084,736 (0.28%)
Time: 2462.8s | Peak VRAM: 6187MB | Trainable: 4,364,288 (0.28%)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.301016,0.435943,3,0.922000,0.922114


TEST SET RESULTS (LoRA, 5000 samples): {'eval_loss': 0.4359428584575653, 'eval_accuracy': 0.922, 'eval_f1': 0.9221140351780375}


In [68]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, EarlyStoppingCallback, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 1. 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NormalFloat4 — better accuracy than plain int4
    bnb_4bit_compute_dtype=torch.bfloat16,  # computation happens in bf16 even though storage is 4-bit
    bnb_4bit_use_double_quant=True,          # quantizes the quantization constants too, saves a bit more memory
)

# 2. Load model in 4-bit
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    num_labels=4, id2label=id2label, label2id=label2id,
    quantization_config=bnb_config,
    device_map={"":0}
)

# 3. Prepare the quantized model for training (required step for QLoRA specifically)
model = prepare_model_for_kbit_training(model)

# 4. Apply LoRA — same config as before
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()  # should still show ~0.28%

# 5. Training args — new output_dir
training_args = TrainingArguments(
    output_dir="./results/qlora-500",
    eval_strategy="epoch", save_strategy="no", load_best_model_at_end=False,
    per_device_train_batch_size=4, per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, num_train_epochs=5,
    learning_rate=2e-4, weight_decay=0.01, logging_steps=10,
    metric_for_best_model="f1", seed=42, report_to="none",
    bf16=True, gradient_checkpointing=True,
)

# 6. Trainer
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_train_500, eval_dataset=tokenized_val_500,
    data_collator=data_collator, processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

reset_vram()
start = time.time()
trainer.train()
elapsed = time.time() - start

peak_vram = get_peak_vram_mb()
trainable, total = count_trainable_params(model)
print(f"Time: {elapsed:.1f}s | Peak VRAM: {peak_vram:.0f}MB | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

test_results = trainer.evaluate(tokenized_test)
print("TEST SET RESULTS (QLoRA, 500 samples):", test_results)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 4,364,288 || all params: 1,548,084,736 || trainable%: 0.2819


[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,4.724484,0.557907,0.780000,0.769638
2,1.229291,0.378422,0.840000,0.838247
3,0.430971,0.380954,0.900000,0.898333
4,0.048199,0.538077,0.880000,0.879808
5,0.004337,0.552955,0.880000,0.881438


Trainable: 4,364,288 / Total: 892,986,880 (0.49%)
Time: 302.8s | Peak VRAM: 7531MB | Trainable: 4,364,288 (0.49%)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.004337,0.575457,5,0.872667,0.873354


TEST SET RESULTS (QLoRA, 500 samples): {'eval_loss': 0.5754573345184326, 'eval_accuracy': 0.8726666666666667, 'eval_f1': 0.8733542940448261}


In [69]:
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    num_labels=4, id2label=id2label, label2id=label2id,
    quantization_config=bnb_config,
    device_map={"": 0}
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./results/qlora-5000",
    eval_strategy="epoch", save_strategy="no", load_best_model_at_end=False,
    per_device_train_batch_size=4, per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, num_train_epochs=3,
    learning_rate=2e-4, weight_decay=0.01, logging_steps=20,
    metric_for_best_model="f1", seed=42, report_to="none",
    bf16=True, gradient_checkpointing=True,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_train_5000, eval_dataset=tokenized_val_5000,
    data_collator=data_collator, processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

reset_vram()
start = time.time()
trainer.train()
elapsed = time.time() - start

peak_vram = get_peak_vram_mb()
trainable, total = count_trainable_params(model)
print(f"Time: {elapsed:.1f}s | Peak VRAM: {peak_vram:.0f}MB | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

test_results = trainer.evaluate(tokenized_test)
print("TEST SET RESULTS (QLoRA, 5000 samples):", test_results)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 4,364,288 || all params: 1,548,084,736 || trainable%: 0.2819


[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.103348,0.199899,0.934000,0.933759
2,0.869331,0.238825,0.922000,0.921823
3,0.456445,0.316855,0.934000,0.933875


Trainable: 4,364,288 / Total: 892,986,880 (0.49%)
Time: 1995.4s | Peak VRAM: 3174MB | Trainable: 4,364,288 (0.49%)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.456445,0.382563,3,0.929333,0.929294


TEST SET RESULTS (QLoRA, 5000 samples): {'eval_loss': 0.3825625777244568, 'eval_accuracy': 0.9293333333333333, 'eval_f1': 0.9292938590531378}
